# Goodreads Machine Learning - Group Project


--------------------------------------
## 1. Loading Data
--------------------------------------

In [95]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

CSV_RAW_PATH = "../data/raw/books.csv"

In [96]:
df = pd.read_csv(CSV_RAW_PATH, index_col="bookID", on_bad_lines='warn')


C:\Users\hugom\AppData\Local\Temp\ipykernel_39744\1247233899.py:1: ParserWarning: Skipping line 3350: expected 12 fields, saw 13
Skipping line 4704: expected 12 fields, saw 13
Skipping line 5879: expected 12 fields, saw 13
Skipping line 8981: expected 12 fields, saw 13

  df = pd.read_csv(CSV_RAW_PATH, index_col="bookID", on_bad_lines='warn')


Upon inspection, the 4 errors came from the fact that the author string contained a comma "," which was interpreted as a new column and spilled the author name into the nex column when creating the .csv, thus resulting in 13 columns instead of 12

In [97]:
# the 4 errors were manualy fixed (joining the 2 columns with "," for these 4 error rows)
CSV_CLEANED_PATH = "../data/processed/books-hugo.csv" 
df = pd.read_csv(CSV_CLEANED_PATH, on_bad_lines='warn')

In [98]:
df.head()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


------------------------------------------
## 2. Cleaning Data
------------------------------------------

In [99]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11127 entries, 0 to 11126
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11127 non-null  int64  
 1   title               11127 non-null  str    
 2   authors             11127 non-null  str    
 3   average_rating      11127 non-null  float64
 4   isbn                11127 non-null  str    
 5   isbn13              11127 non-null  int64  
 6   language_code       11127 non-null  str    
 7     num_pages         11127 non-null  int64  
 8   ratings_count       11127 non-null  int64  
 9   text_reviews_count  11127 non-null  int64  
 10  publication_date    11127 non-null  str    
 11  publisher           11127 non-null  str    
dtypes: float64(1), int64(5), str(6)
memory usage: 1.0 MB


### **observations :**
- It looks like there is a leading space for the " num_pages" column name
- isbn13 as int64 seems like the wrong type for and identifier, but not a big deal because we will probably drop these columns
- publication_date as string is not ideal, should be parsed to date

In [100]:
# check if bookID, ISBN, ISBN13 are all uniques
cols = ["bookID", "isbn", "isbn13"]
{col: df[col].is_unique for col in cols}

{'bookID': True, 'isbn': True, 'isbn13': True}

In [101]:
# since all are unique, we can select any for indexing, lets select bookID
df.set_index("bookID", inplace=True)

In [102]:
# fix column names
df.columns = df.columns.str.strip()

In [103]:
# test if any of the values contains leading/trailing spaces
for col in df.columns:
    if df[col].dtype == "str":
        has_spaces = df[col].dropna().ne(df[col].dropna().str.strip()).any()
        if has_spaces:
            print(f"Leading/trailing spaces detected in column: {col}")
        else:
            print(f"no spaces detected on column : {col}")



Leading/trailing spaces detected in column: title
no spaces detected on column : authors
no spaces detected on column : isbn
no spaces detected on column : language_code
no spaces detected on column : publication_date
no spaces detected on column : publisher


In [104]:
# fix publication_date, passing format to parse
pub_raw = df["publication_date"].copy()
df["publication_date"] = pd.to_datetime(
    df["publication_date"],
    format="%m/%d/%Y",
    errors="coerce", # if the format is not correct, set to NaT
)
# check original values where the conversion failed
pub_raw.loc[df["publication_date"].isna()]

bookID
31373    11/31/2000
45531     6/31/1982
Name: publication_date, dtype: str

these dates are impossible, lets check ISBN code and look for the information on the internet since there is only 2 books with invalid dates

In [105]:
# check ISBN13 code
df.loc[df.publication_date.isna(), ["isbn","isbn13","title"]]

,isbn,isbn13,title
bookID,,,
31373,0553575104,9780553575101,In Pursuit of the Proper Sinner (Inspector Lyn...
45531,2070323285,9782070323289,Montaillou village occitan de 1294 à 1324


According to current goodreads.com: 
- In Pursuit of the Proper Sinner was puplished in October 31st, 2000
- Montaillou  village occitan de 1294 à 1324 was published in June 30, 1982

One day is wrong by 1 day, the other by 1 month. 

These errors could be isolated, or might be a symptom of a larger issue with dates. Some of the parsable date might still be wrong. Goodreads.com does not provide a public API to check the data programmatically (an unofficial one exist on Apify), but there are other options to check the publishing date with Open Library or Google Books API.  

Lets apply a target fix:

In [106]:
# fix the dates by index
df.loc[31373, "publication_date"] = pd.to_datetime("2000-10-31")
df.loc[45531, "publication_date"] = pd.to_datetime("1982-06-30")

In [107]:
df.describe()

,average_rating,isbn13,num_pages,ratings_count,text_reviews_count,publication_date
count,11127.000000,1.112700e+04,11127.000000,1.112700e+04,11127.000000,11127
mean,3.933631,9.759888e+12,336.376921,1.793641e+04,541.854498,2000-08-27 22:58:00.614721
min,0.000000,8.987060e+09,0.000000,0.000000e+00,0.000000,1900-01-01 00:00:00
25%,3.770000,9.780345e+12,192.000000,1.040000e+02,9.000000,1998-07-16 12:00:00
50%,3.960000,9.780586e+12,299.000000,7.450000e+02,46.000000,2003-03-01 00:00:00
75%,4.135000,9.780873e+12,416.000000,4.993500e+03,237.500000,2005-09-30 00:00:00
max,5.000000,9.790008e+12,6576.000000,4.597666e+06,94265.000000,2020-03-31 00:00:00
std,0.352445,4.428964e+11,241.127305,1.124794e+05,2576.176608,NaN


At first glance, we can see that there are rows with num_pages, ratings_count and text_reviews_count equal to 0  
➜ should be investigated and cleaned

In [108]:
df.isna().sum()

title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
num_pages             0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

Check for other bad/empty data :

In [109]:
# check for bad data
bad_inputs = ["", "nan", "None", "N/A", "null", "-"]
bad_results = {}
for col in df.columns:
    mask = df[col].astype(str).str.strip().str.lower().isin(bad_inputs)
    bad_results[col] = int(mask.sum())
bad_results


{'title': 0,
 'authors': 0,
 'average_rating': 0,
 'isbn': 0,
 'isbn13': 0,
 'language_code': 0,
 'num_pages': 0,
 'ratings_count': 0,
 'text_reviews_count': 0,
 'publication_date': 0,
 'publisher': 0}

➜ no strictly missing values

In [110]:
print("num_pages zero count :",(df["num_pages"] == 0).sum())
print("ratings_count zero count :",(df["ratings_count"] == 0).sum())
print("text_reviews_count zero count :",(df["text_reviews_count"] == 0).sum())

num_pages zero count : 76
ratings_count zero count : 81
text_reviews_count zero count : 625


Need to decide what to do with these :

num_pages = 0 ➜ encode average/median?  
ratings_count = 0 ➜ drop  
text_reviews_count = 0 ➜ could be an actual signal, keep

In [111]:
print("average_rating negative count :",(df["average_rating"] < 0).sum())

average_rating negative count : 0


In [112]:
print("ratings_count zero with average_rating not zero:",((df["ratings_count"] == 0) & (df["average_rating"] != 0)).sum())

ratings_count zero with average_rating not zero: 55


This is an issue, a book shouldn't have a positive rating if the ratings count is zero

In [113]:
df[(df["ratings_count"] == 0) & (df["average_rating"] != 0)].head()

,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
bookID,,,,,,,,,,,
797,Lonely Planet Londres,Lonely Planet/Sarah Johnstone/Tom Masters,4.03,8408064762,9788408064763,spa,480,0,0,2006-05-01,Geoplaneta
1658,American Government: Continuity and Change Al...,Karen O'Connor/Larry J. Sabato,2.83,0321317106,9780321317100,eng,664,0,0,2005-03-11,Longman Publishing Group
1664,Essentials of American and Texas Government: C...,Karen O'Connor/Larry J. Sabato,3.50,0321365208,9780321365200,eng,854,0,0,2005-07-29,Longman Publishing Group
2034,Comoediae 1: Acharenses/Equites/Nubes/Vespae/P...,Aristophanes/F.W. Hall/W.M. Geldart,5.00,0198145047,9780198145042,grc,364,0,0,1922-02-22,Oxford University Press USA
2411,Melville and the politics of identity: From *K...,Julian Markels,3.33,0252063023,9780252063022,eng,164,0,0,1993-07-01,University of Illinois Press


**Hypothesis:** 
From this snippet, we can hypothesize that the reason why these books have a rating > 0 while ratings_count = 0 is because books share the same rating accross original & translated versions. Lets test this hypothesis

In [114]:
# check if there are other books with the same title as the greek version of bookID 2034
title_2034 = df.loc[2034, "title"]
df["title"].eq(title_2034).sum()

np.int64(1)

In [118]:
# top 10 duplicated titles
vc = df["title"].value_counts()
#top_titles = vc[vc > 1].head(10).index
vc[vc > 1].head(10)
#df[df["title"].isin(top_titles)].sort_values("title")

title
The Iliad                     9
The Brothers Karamazov        9
Anna Karenina                 8
The Odyssey                   8
'Salem's Lot                  8
Gulliver's Travels            8
The Picture of Dorian Gray    7
A Midsummer Night's Dream     7
Treasure Island               6
Collected Stories             6
Name: count, dtype: int64

In [122]:
df.loc[df["title"]=="The Iliad",["title","isbn","language_code","average_rating"]]

,title,isbn,language_code,average_rating
bookID,,,,
1371,The Iliad,0140275363,eng,3.86
1374,The Iliad,0374529051,en-US,3.86
1376,The Iliad,0140447946,eng,3.86
1377,The Iliad,0451527372,en-US,3.86
1796,The Iliad,1857150600,eng,3.86
12254,The Iliad,0143059289,eng,3.86
22221,The Iliad,0471377589,eng,3.86
32780,The Iliad,1904633382,eng,3.86
32782,The Iliad,0753453215,eng,3.86
